# Attention-Based Emotion Localization — XAI Visualization

This notebook demonstrates the **novel XAI component** of our Multimodal SER system:

1. **Audio Attention Heatmap** — Which time frames in the speech were most emotionally salient?
2. **Text Token Attention** — Which words drove the emotion prediction?
3. **Emotion Drift Plot** — How does emotion evolve across a conversation?

Load a trained checkpoint and run inference on individual utterances to see *why* the model predicted a given emotion.


In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from transformers import AutoTokenizer

from src.models.ser_model import build_model
from src.data.preprocessing import process_audio
from src.data.dataset import EMOTION_TO_IDX, IDX_TO_EMOTION
from src.utils.visualization import (
    plot_audio_attention,
    plot_text_attention,
    render_text_attention_html,
    plot_emotion_drift,
)

# ── Config
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = '../checkpoints/fold5_best.pt'  # Change fold as needed
CSV_PATH = '../data/processed/iemocap_6class.csv'

print(f'Device: {DEVICE}')

## Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config['model']['roberta_model'])

model = build_model(config).to(DEVICE)
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded from: {CHECKPOINT}')
print(f'Checkpoint epoch: {ckpt["epoch"]} | Val WF1: {ckpt["metrics"]["weighted_f1"]:.4f}')

## Inference Helper

In [ ]:
@torch.no_grad()
def predict(audio_path: str, text: str):
    """Run model inference and return predictions + attention weights."""
    cfg = config['data']
    audio = process_audio(
        audio_path, cfg['sample_rate'], cfg['n_mels'],
        cfg['n_fft'], cfg['hop_length'], cfg['fmax'], cfg['max_audio_seconds']
    ).unsqueeze(0).to(DEVICE)  # [1, 1, 128, T]

    encoding = tokenizer(
        text, max_length=config['model']['max_text_len'],
        padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(DEVICE)
    attn_mask = encoding['attention_mask'].to(DEVICE)

    outputs = model(audio, input_ids, attn_mask)

    pred_idx = outputs['emotion_logits'].argmax(dim=-1).item()
    pred_emotion = IDX_TO_EMOTION[pred_idx]
    vad = outputs['vad_scores'][0].cpu().numpy()
    audio_attn = outputs['audio_attn'][0].cpu().numpy()
    token_attn = outputs['token_attn'][0].cpu().numpy()

    # Get actual tokens (not padded)
    token_ids = encoding['input_ids'][0]
    mask = encoding['attention_mask'][0].bool()
    tokens = tokenizer.convert_ids_to_tokens(token_ids[mask])
    token_attn_clean = token_attn[mask.cpu().numpy().astype(bool)]

    return {
        'pred_emotion': pred_emotion,
        'vad': vad,
        'audio_attn': audio_attn,
        'tokens': tokens,
        'token_attn': token_attn_clean,
    }

## 1. Pick a Sample Utterance

In [ ]:
df = pd.read_csv(CSV_PATH)
# Pick an angry utterance from session 5 (test session)
sample = df[(df['session'] == 5) & (df['emotion'] == 'ang')].sample(1, random_state=42).iloc[0]

print(f'Utterance ID : {sample["utterance_id"]}')
print(f'Text         : {sample["text"]}')
print(f'True emotion : {sample["emotion"].upper()}')
print(f'VAD (true)   : V={sample["valence"]:.2f}, A={sample["arousal"]:.2f}, D={sample["dominance"]:.2f}')
print(f'Audio path   : {sample["audio_path"]}')

## 2. Run Inference

In [ ]:
result = predict(sample['audio_path'], sample['text'])

print(f'Predicted emotion : {result["pred_emotion"].upper()}')
print(f'VAD (predicted)   : V={result["vad"][0]:.2f}, A={result["vad"][1]:.2f}, D={result["vad"][2]:.2f}')
correct = '✓' if result['pred_emotion'] == sample['emotion'] else '✗'
print(f'Correct?          : {correct}')

## 3. Audio Attention Heatmap

The overlay highlights *which time frames* the model focused on when predicting the emotion. Red = high attention (emotionally salient), dark = low attention.

In [ ]:
cfg = config['data']
fig = plot_audio_attention(
    audio_path=sample['audio_path'],
    attn_weights=result['audio_attn'],
    predicted_emotion=result['pred_emotion'],
    true_emotion=sample['emotion'],
    sample_rate=cfg['sample_rate'],
    hop_length=cfg['hop_length'],
    n_mels=cfg['n_mels'],
    n_fft=cfg['n_fft'],
    fmax=cfg['fmax'],
    save_path=f'../logs/{sample["utterance_id"]}_audio_attn.png',
)
plt.show()

## 4. Text Token Attention

Which words in the transcript most influenced the emotion prediction? Darker red = higher attention.

In [ ]:
# Rich HTML heatmap
html = render_text_attention_html(
    tokens=result['tokens'],
    attn_weights=result['token_attn'],
    predicted_emotion=result['pred_emotion'],
)
display(HTML(html))

In [ ]:
# Bar chart version
fig = plot_text_attention(
    tokens=result['tokens'],
    attn_weights=result['token_attn'],
    predicted_emotion=result['pred_emotion'],
    save_path=f'../logs/{sample["utterance_id"]}_text_attn.png',
)
plt.show()

## 5. Emotion Drift — Full Dialog

Trace the emotion trajectory across all utterances in the same dialog conversation. Red dashed lines = emotion transition events.

In [ ]:
# Get all utterances from same dialog
dialog_df = df[df['dialog'] == sample['dialog']].reset_index(drop=True)
print(f'Dialog: {sample["dialog"]} | Utterances: {len(dialog_df)}')

# Predict all utterances in dialog
dialog_preds = []
for _, row in dialog_df.iterrows():
    r = predict(row['audio_path'], row['text'])
    dialog_preds.append(r['pred_emotion'])

# Compute transitions
transitions = []
for i in range(1, len(dialog_preds)):
    if dialog_preds[i] != dialog_preds[i-1]:
        transitions.append({
            'from': dialog_preds[i-1],
            'to': dialog_preds[i],
            'position': i,
            'at_utterance': dialog_df.iloc[i]['utterance_id'],
        })

print(f'Detected {len(transitions)} emotion drift events')

fig = plot_emotion_drift(
    utterance_ids=dialog_df['utterance_id'].tolist(),
    emotions=dialog_preds,
    transitions=transitions,
    dialog_name=sample['dialog'],
    save_path=f'../logs/{sample["dialog"]}_drift.png',
)
plt.show()

## Summary

The three visualizations above together form the **Attention-Based Emotion Localization** (XAI) system:

| Visualization | What it shows | XAI insight |
|---|---|---|
| Audio heatmap | Frame-level attention on spectrogram | *When* was the emotion expressed? |
| Text heatmap | Token-level attention on transcript | *Which words* caused the emotion? |
| Drift plot | Emotion sequence across dialog | *How* did emotion evolve over time? |

This makes the model **interpretable and trustworthy** — a key novelty over standard black-box SER classifiers.
